# Odd-torsion search for $G(\mathbb{Q}^3, d)$

Direct (vector) implementation of the method from `odd_torsion.py`: for an integer $d$ and denominator $D$ we build
$S=\{v\in\mathbb{Z}^3: |v|^2=d D^2\}$, the Heuberger matrix on $S$, and search the saturation of its column lattice
for a vector of odd augmentation. A find proves $\chi(\mathbb{Q}^3,d)=4$.

Every run saves into `Q_3/runs/d{d}_D{D}/` (the folder is renamed right after saving, gaining a leading `+`
if torsion was found — `Q_3/runs/+d{d}_D{D}/`):

- `vectors.txt` — `index: vector`, one index per vector of $S$ (v and $-v$ get separate indices);
- `matrix.txt` — the Heuberger matrix's relation rows as signed indices into `vectors.txt`:
  backtracks as `+i +j`, parallelograms as `+i +j -k -l`;
- `odd_torsion_vector.txt` — the found odd-torsion vector, lines `"index" x "count"` (only if found;
  `count` carries the actual signed coefficient).

`verify_independent.py` re-checks a saved run folder from scratch (does not import `odd_torsion.py`) —
see the last section of this notebook.

In [1]:
import sys, os
sys.path.append(os.getcwd())

from odd_torsion import search, search_and_save, save_run

## Scanning over $d \in T$ at a fixed $D$

The cell below defines $T$ (even squarefree $d$ not covered by the triangle family, see the next cell)
and loops `search_and_save(d, D)` over it at a fixed $D=9$, saving every attempt.

In [2]:
S = sorted([2*(a**2 + a*b + b**2) for a in range(-100, 100) for b in range(-100, 100)])
S = [ZZ(x).squarefree_part() for x in S]
T = [ZZ(x).squarefree_part() for x in range(2, 1000, 2)]
T = sorted([x for x in set(T) - set(S) if x % 2 == 0])
#T = sorted([x for x in set(T) if x % 2 == 0])
print(*T, sep=' ')

10 22 30 34 46 58 66 70 82 94 102 106 110 118 130 138 142 154 166 170 174 178 190 202 210 214 226 230 238 246 262 274 282 286 290 298 310 318 322 330 334 346 354 358 370 374 382 390 394 406 410 418 426 430 442 454 462 466 470 478 498 502 506 510 514 526 530 534 538 562 570 574 586 590 598 606 610 622 634 638 642 646 658 670 678 682 690 694 706 710 714 718 730 742 754 766 770 778 782 786 790 802 814 822 826 830 838 858 862 870 874 886 890 894 898 902 910 922 930 934 946 958 966 970 982 986 994


In [ ]:
for d in T:
    if d < 140:
        continue
    D = 9
    result, path = search_and_save(d, D, orbit_order="greedy_rank")
    print()
    print("folder:", path)
    print("found:", result["found"])
    if result["found"]:
        print("support (vector, coefficient):")
        for v, c in result["support"]:
            print("  ", v, c)
    print('-'*80)
    print('-'*80)

d=142, D=9: N=d*D^2=11502, |S|=816
    greedy_rank: orbit 0 (rank(GF1000000007) -> 39)
    greedy_rank: orbit 4 (rank(GF1000000007) -> 84)
    greedy_rank: orbit 15 (rank(GF1000000007) -> 132)
  appending orbits in order...
  orbit 1/17 (+48, active 48/816): not found
  orbit 2/17 (+48, active 96/816): not found
  orbit 3/17 (+48, active 144/816): not found
  orbit 4/17 (+48, active 192/816): not found
  orbit 5/17 (+48, active 240/816): not found


## Independent verification

`verify_independent.py` re-parses a saved run folder's plain-text files from scratch
(`vectors.txt`, `matrix.txt`, `odd_torsion_vector.txt`) and independently re-checks
Proposition 3.1: odd augmentation, zero boundary, and — the check this was built for —
that appending the odd-torsion vector as an extra column to the Heuberger matrix does
**not** increase its rank (i.e. the vector lies in $\mathrm{colspan}_\mathbb{Q}(H_F)$).
It deliberately does not import `odd_torsion.py`, so a bug shared between the search
code and the checker cannot silently validate a wrong answer.

In [ ]:
from verify_independent import verify_folder

# path to any saved run folder, e.g. the one produced by the single-run cell above
verify_folder(path)